# GA4 Gözlemlenmiş LTV Analizi

Bu notebook, GA4 BigQuery verilerinden `user_id` bazlı gözlemlenmiş LTV ile aylık e-ticaret ve dijital performans metriklerini hesaplar. Eşik değeri veya tahmin modeli kullanılmaz.

**Akış:** bağlantıyı doğrula → dry run ile taramayı gör → temel tabloyu oluştur → LTV, aylık KPI, MoM/YoY ve temel kohort analizlerini çalıştır → tek dosyalık HTML raporu indir.

## 1. Paketi yükle

In [ ]:
%pip install -q --upgrade "ga4-ltv-toolkit @ git+https://github.com/yasinsariyildizz/ga4_ltv_analyzer.git@main"

GitHub deposu henüz yayımlanmadıysa üstteki hücre yerine `ga4_ltv.py` dosyasını Colab'a yükleyip aynı `from ga4_ltv import LTVAnalyzer` satırını kullanabilirsiniz.

## 2. Google hesabını doğrula

In [ ]:
from google.colab import auth
auth.authenticate_user()

## 3. Girişler — yalnızca bu dört alanı değiştirin

In [ ]:
PROJECT_ID = "client-project-id" # @param {type:"string"}
DATASET_ID = "analytics_123456789" # @param {type:"string"}
TABLE_ID = "events_*" # @param {type:"string"}
OUTPUT_DATASET_ID = "ga4_ltv" # @param {type:"string"}

## 4. Analizi başlat ve bağlantıyı kontrol et

In [ ]:
from ga4_ltv import LTVAnalyzer

analysis = LTVAnalyzer(
    project_id=PROJECT_ID,
    dataset_id=DATASET_ID,
    table_id=TABLE_ID,
    output_dataset_id=OUTPUT_DATASET_ID,
)

analysis.validate()

## 5. Dry run — önce tahmini taramayı inceleyin

Bu hücre hiçbir tablo yazmaz. Sonuç beklediğinizden büyükse devam etmeden önce kaynak tablo desenini kontrol edin.

In [ ]:
dry_run_result = analysis.dry_run()
dry_run_result

## 6. Satın alma ve iade temel tablosunu oluştur

Bu adımdan itibaren BigQuery sorguları gerçekten çalışır ve sonuçlar `OUTPUT_DATASET_ID` içine yazılır.

In [ ]:
base_result = analysis.create_base_table()
base_result["data_quality"]

## 7. LTV analizini çalıştır

Müşteri özetleri, genel istatistikler, LTV dilimleri, gelir yoğunlaşması ve yorum notları üretilir.

In [ ]:
ltv_result = analysis.ltv_analysis()

## 8. Aylık e-ticaret ve dijital performansı çalıştır

ARPU, ARPPU, AOV, net AOV, oturum başına gelir, satın alma sıklığı, etkileşim, funnel ve iade metrikleri oluşturulur. Eksiksiz takvim ayları için MoM ve YoY değişimleri hesaplanır.

In [ ]:
monthly_result = analysis.monthly_metrics_analysis()

## 9. Temel kohort özetini çalıştır

Müşteriler ilk satın alma ayına göre gruplanır; kohort büyüklüğü, bugüne kadar gözlemlenen LTV ve tekrar oranı özetlenir. Eşit yaşta ayrıntılı kohort analizi bu modülün kapsamı dışındadır.

In [ ]:
cohort_result = analysis.cohort_analysis()

## 10. Tek dosyalık HTML raporu oluştur ve indir

In [ ]:
DOWNLOAD_HTML = True # @param {type:"boolean"}

dashboard_path = analysis.generate_dashboard("ga4_ltv_dashboard.html")

if DOWNLOAD_HTML:
    from google.colab import files
    files.download(dashboard_path)

dashboard_path

## Çıktıyı yorumlarken

- Bu sonuç gelecekteki geliri tahmin etmez; seçilen veri geçmişinde gözlenen net geliri gösterir.
- Ortalama ile ortancayı birlikte okuyun. Aradaki büyük fark, yüksek değerli az sayıda müşterinin ortalamayı yükselttiğini gösterebilir.
- E-ticaret ARPU bu pakette net satın alma geliri / aktif kullanıcı; AOV brüt satın alma geliri / sipariş olarak tanımlanır.
- MoM ve YoY yalnızca eksiksiz takvim ayları için hesaplanır; karşılaştırma ayı yoksa değer boş kalır.
- Temel kohortlar aynı gözlem yaşında değildir. Eski bir kohortun yüksek LTV'si daha uzun gözlem süresinden kaynaklanabilir.
- `user_id` kapsamı düşükse LTV sonuçları tüm satın alanları değil, kimliği belirlenebilen satın alanları temsil eder. Aylık metrikler gerektiğinde `user_pseudo_id` kullanır.
- ROAS, CPA ve CAC için ayrıca reklam maliyeti kaynağı gerekir; bu dört girişten türetilmez.